# AC14 Master Journey: Structured Input → Validated System

This is the **primary end-to-end journey notebook** for AC14.

It walks the full AC14 pipeline from realistic structured input through blueprint
derivation, packet compilation, component generation, recomposition proof, and
acceptance review. The empirical comparison gate appears as the final planned
phase.

Subject: `examples/support_ticket_digest` — a multi-component support-ticket
triage workflow.

## Journey Contract

| Field | Value |
|-------|-------|
| Journey ID | `ac14_master_journey` |
| Subject | `examples/support_ticket_digest` |
| Notebook mode | mixed — live back-half, fixture front-half, planned empirical gate |
| Related docs | `docs/AC14_ROADMAP.md`, `docs/AC14_ANTI_DRIFT_DOCTRINE.md`, `CLAUDE.md` |
| Related plans | `docs/plans/38_empirical_comparison_gate.md`, `docs/plans/82_front_half_first_empirical_contract.md` |

### Full Journey

```
structured input
  → [Phase 1] inspect input fields and schema
  → [Phase 2] front-half: discovery → dependency planning → draft blueprint → freeze
  → [Phase 3] load and validate frozen blueprint (B1)
  → [Phase 4] compile bounded component packets (B2)
  → [Phase 5] generate deterministic components
  → [Phase 6] run recomposition proof
  → [Phase 7] acceptance review against requirements
  → [Phase 8] empirical comparison gate (front-half-first)
```

### Execution Modes

| Phase | Status | Execution Mode | LLM required? |
|-------|--------|---------------|---------------|
| 1. Input inspection | proven | live | no |
| 2. Front-half overview | proven | fixture | no (shipped blueprint is the fixture) |
| 3. Blueprint validation | proven | live | no |
| 4. Packet compilation | proven | live | no |
| 5. Component generation | proven | live | no (deterministic) |
| 6. Recomposition proof | proven | live | no |
| 7. Acceptance review | proven | dry_run | no (reference generator) |
| 8. Empirical gate | planned | planned | yes (smoke runner) |

In [1]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / 'pyproject.toml').exists()
            and (candidate / 'docs' / 'plans' / 'CLAUDE.md').exists()
        ):
            return candidate
    raise RuntimeError('Could not locate the AC14 repo root.')


REPO_ROOT = find_repo_root(Path.cwd())
EXAMPLE_DIR = REPO_ROOT / 'examples' / 'support_ticket_digest'
BLUEPRINT_DIR = EXAMPLE_DIR / 'blueprint'
INPUT_FILE = EXAMPLE_DIR / 'input' / 'realistic_ticket_batch.json'

assert EXAMPLE_DIR.exists(), f'Missing example: {EXAMPLE_DIR}'
assert BLUEPRINT_DIR.exists(), f'Missing blueprint: {BLUEPRINT_DIR}'
assert INPUT_FILE.exists(), f'Missing input: {INPUT_FILE}'

print(f'Repo root: {REPO_ROOT}')
print(f'Blueprint: {BLUEPRINT_DIR}')
print(f'Input:     {INPUT_FILE}')

Repo root: /home/brian/projects/ac14
Blueprint: /home/brian/projects/ac14/examples/support_ticket_digest/blueprint
Input:     /home/brian/projects/ac14/examples/support_ticket_digest/input/realistic_ticket_batch.json


---
## Phase 1: Input Inspection

**Input:** `examples/support_ticket_digest/input/realistic_ticket_batch.json`  
**Output:** field inventory — record count, field names, value shapes  
**Status:** `proven`  
**Execution mode:** `live`  

### Acceptance Criteria
1. Input file loads without error.
2. Record count is explicit.
3. Field names and representative values are visible.
4. Any structural anomalies (missing fields, mixed types) are surfaced.

In the full front-half pipeline this phase feeds `make discover-input`, which
records field summaries, infers schemas, and flags open concerns before blueprint
freeze.

In [2]:
# INPUT
input_file = INPUT_FILE          # ← swap to any .json batch file to experiment

# Load all records and show the first one so you can see the raw schema.
records = json.loads(input_file.read_text())
assert isinstance(records, list) and len(records) > 0

print(f'{len(records)} records. First record:')
records[0]                       # OUTPUT: first ticket dict

3 records. First record:


{'ticket_id': 'SUP-10421',
 'customer_id': 'cust-7781',
 'subject': 'SSO login fails for finance team after IdP certificate rotation',
 'body': 'Since the certificate rollover at 08:15 UTC, finance users see a blank callback page after Okta login. Payroll close is tonight and four users are blocked.',
 'product_area': 'authentication',
 'channel': 'email',
 'auth_method': 'sso',
 'reported_priority': 'high',
 'tags': ['sso', 'okta', 'finance', 'certificate'],
 'opened_at': '2026-03-31T08:32:00Z'}

In [3]:
# INPUT
# records  (from previous cell)

from collections import defaultdict

# Build a map of field → observed Python types across all records.
# If a field is sometimes str and sometimes NoneType that signals schema drift —
# the blueprint will need to handle it explicitly.
field_types: dict[str, set[str]] = defaultdict(set)
for record in records:
    for k, v in record.items():
        field_types[k].add(type(v).__name__)

# Flag records that are missing fields present in record 0.
# Missing fields mean the input schema is not guaranteed — important for
# blueprint planning (discovery will flag these as open concerns).
baseline_fields = set(records[0].keys())
anomalies = [
    {'index': i, 'missing': sorted(baseline_fields - set(r.keys()))}
    for i, r in enumerate(records)
    if set(r.keys()) != baseline_fields
]

phase1_result = {
    'record_count': len(records),
    'field_count': len(baseline_fields),
    'structural_anomaly_count': len(anomalies),
    'fields': {
        field: {'types_observed': sorted(types), 'sample_value': records[0].get(field)}
        for field, types in sorted(field_types.items())
    },
    'anomalies': anomalies[:5],
}

phase1_result  # OUTPUT: field inventory

{'record_count': 3,
 'field_count': 10,
 'structural_anomaly_count': 0,
 'fields': {'auth_method': {'types_observed': ['str'], 'sample_value': 'sso'},
  'body': {'types_observed': ['str'],
   'sample_value': 'Since the certificate rollover at 08:15 UTC, finance users see a blank callback page after Okta login. Payroll close is tonight and four users are blocked.'},
  'channel': {'types_observed': ['str'], 'sample_value': 'email'},
  'customer_id': {'types_observed': ['str'], 'sample_value': 'cust-7781'},
  'opened_at': {'types_observed': ['str'],
   'sample_value': '2026-03-31T08:32:00Z'},
  'product_area': {'types_observed': ['str'],
   'sample_value': 'authentication'},
  'reported_priority': {'types_observed': ['str'], 'sample_value': 'high'},
  'subject': {'types_observed': ['str'],
   'sample_value': 'SSO login fails for finance team after IdP certificate rotation'},
  'tags': {'types_observed': ['list'],
   'sample_value': ['sso', 'okta', 'finance', 'certificate']},
  'ticket_id'

---
## Phase 2: Front-Half Overview (Discovery → Freeze)

**Input:** structured input records + project context  
**Output:** `examples/support_ticket_digest/blueprint/` — six-file frozen bundle  
**Status:** `proven`  
**Execution mode:** `fixture` (the shipped blueprint is the real output of a prior front-half run)  

### Acceptance Criteria
1. All six required blueprint files exist.
2. The metadata blueprint ID is explicit.
3. The component list matches what discovery and planning produced.

### What live mode does

In live mode the front half runs:
```
make discover-input          → discovery_artifact.json
make plan-dependencies       → dependency_plan.json
make probe-dependencies      → dependency_execution_artifact.json
make draft-blueprint-plan    → draft_blueprint_plan.json
make materialize-draft-bundle → 6-file draft bundle
make decide-freeze           → freeze_decision.json + approved/ bundle
```

Each step requires LLM calls. The shipped blueprint is the committed fixture
output of that pipeline run on `support_ticket_digest`. To run live:
```bash
make front-half-acceptance \
  REALISTIC_INPUT=examples/support_ticket_digest/input/realistic_ticket_batch.json \
  OUTPUT=.ac14_out/front_half \
  REQUIREMENTS="preserve ticket meaning keep packets bounded" \
  PACKAGES="pydantic"
```

### Promotion path
When the front-half-first empirical gate validates that AC14 front-half quality
materially outperforms monolithic drafting, this phase should be promoted to
`live` mode using the structured-spec front-half runner.

In [4]:
# INPUT
blueprint_dir = BLUEPRINT_DIR    # ← swap to any blueprint/ directory to experiment
import yaml

REQUIRED_FILES = ['metadata.yaml', 'schemas.yaml', 'components.yaml',
                  'architecture.yaml', 'validation.yaml', 'fixtures.yaml']

# Verify all six required files exist. A missing file means the bundle is
# incomplete and cannot be loaded by AC14's canonical loader.
for fname in REQUIRED_FILES:
    assert (blueprint_dir / fname).exists(), f'Missing: {blueprint_dir / fname}'

# Read the metadata and component list to show what this blueprint covers.
metadata = yaml.safe_load((blueprint_dir / 'metadata.yaml').read_text())
components_raw = yaml.safe_load((blueprint_dir / 'components.yaml').read_text())
schemas_raw = yaml.safe_load((blueprint_dir / 'schemas.yaml').read_text())

phase2_fixture_summary = {
    'blueprint_id': metadata['metadata']['blueprint_id'],
    'component_count': len(components_raw.get('components', [])),
    'schema_count': len(schemas_raw.get('schemas', [])),
    'component_ids': [c['component_id'] for c in components_raw.get('components', [])],
    'files_present': REQUIRED_FILES,
}

phase2_fixture_summary  # OUTPUT: blueprint summary

{'blueprint_id': 'support_ticket_digest_v1',
 'component_count': 5,
 'schema_count': 7,
 'component_ids': ['ticket_parser',
  'issue_classifier',
  'priority_scorer',
  'customer_context_loader',
  'digest_assembler'],
 'files_present': ['metadata.yaml',
  'schemas.yaml',
  'components.yaml',
  'architecture.yaml',
  'validation.yaml',
  'fixtures.yaml']}

---
## Phase 3: Blueprint Loading and B1 Validation

**Input:** `examples/support_ticket_digest/blueprint/` (six-file bundle)  
**Output:** `FrozenBlueprint` model + `ValidationResult` — pass/fail with findings  
**Status:** `proven`  
**Execution mode:** `live`  

### Acceptance Criteria
1. Blueprint loads into the canonical `FrozenBlueprint` model without error.
2. B1 validation passes with zero findings.
3. Component count and schema count match Phase 2.
4. Every component has at least one input port and one output port.

In [5]:
# INPUT
blueprint_dir = BLUEPRINT_DIR    # ← swap to experiment with different blueprints

import sys
sys.path.insert(0, str(REPO_ROOT))
from ac14.loader import load_blueprint_dir
from ac14.validation import validate_blueprint

# Load the six YAML files into the canonical FrozenBlueprint Pydantic model.
# This is the single typed object that all downstream phases consume.
blueprint = load_blueprint_dir(blueprint_dir)

# B1 validation: checks schema references, port bindings, and graph structure.
# It fails loud — if this passes, the blueprint is structurally sound.
b1_result = validate_blueprint(blueprint)
assert b1_result.passed, f'B1 validation failed: {b1_result.findings}'
assert len(blueprint.components) == phase2_fixture_summary['component_count']
assert len(blueprint.schemas)    == phase2_fixture_summary['schema_count']

# Confirm every component has at least one output port (a component with no
# outputs is dead weight in the graph).
for cid, comp in blueprint.components.items():
    assert comp.output_ports, f'{cid} has no output ports'

phase3_result = {
    'b1_passed': b1_result.passed,
    'finding_count': len(b1_result.findings),
    'component_count': len(blueprint.components),
    'schema_count': len(blueprint.schemas),
    'binding_count': len(blueprint.bindings),
    'component_ports': {
        cid: {'inputs': [p.name for p in c.input_ports],
              'outputs': [p.name for p in c.output_ports]}
        for cid, c in blueprint.components.items()
    },
}

phase3_result  # OUTPUT: blueprint structure summary

{'b1_passed': True,
 'finding_count': 0,
 'component_count': 5,
 'schema_count': 7,
 'binding_count': 7,
 'component_ports': {'ticket_parser': {'inputs': ['raw_ticket'],
   'outputs': ['parsed_ticket']},
  'issue_classifier': {'inputs': ['parsed_ticket'],
   'outputs': ['issue_label']},
  'priority_scorer': {'inputs': ['parsed_ticket'],
   'outputs': ['priority_score']},
  'customer_context_loader': {'inputs': ['parsed_ticket'],
   'outputs': ['customer_context']},
  'digest_assembler': {'inputs': ['on_ticket',
    'on_label',
    'on_priority',
    'on_customer_context'],
   'outputs': ['digest_entry', 'digest_store']}}}

---
## Phase 4: Packet Compilation (B2)

**Input:** `FrozenBlueprint`  
**Output:** `PacketBundle` — one bounded `ComponentPacket` per component  
**Status:** `proven`  
**Execution mode:** `live`  

### Acceptance Criteria
1. A packet exists for every component in the blueprint.
2. Each packet contains only its own schemas and bindings — no hidden global context.
3. Upstream and downstream components are referenced by summary, not by full definition.
4. Every packet's local fixture set is non-empty (at least one test case per component).

In [6]:
# INPUT
# blueprint  (FrozenBlueprint from Phase 3)

from ac14.packets import compile_packets

# Compile one bounded ComponentPacket per component. Each packet contains only
# the schemas, bindings, fixtures, and neighbour summaries that component needs —
# no hidden global context leaks through. This is the core boundedness guarantee.
packet_bundle = compile_packets(blueprint)

assert set(packet_bundle.packets.keys()) == set(blueprint.components.keys()), \
    'Packet set does not match component set'

phase4_result = {
    'packet_count': len(packet_bundle.packets),
    'per_component': {
        cid: {
            'local_schemas':    len(pkt.local_schemas),
            'inbound_bindings': len(pkt.inbound_bindings),
            'outbound_bindings':len(pkt.outbound_bindings),
            'local_fixtures':   len(pkt.relevant_scenarios),
            # How many schemas were excluded — higher means better isolation.
            'global_schemas_excluded': len(blueprint.schemas) - len(pkt.local_schemas),
        }
        for cid, pkt in packet_bundle.packets.items()
    },
}

phase4_result  # OUTPUT: per-component packet stats

{'packet_count': 5,
 'per_component': {'ticket_parser': {'local_schemas': 2,
   'inbound_bindings': 0,
   'outbound_bindings': 4,
   'local_fixtures': 2,
   'global_schemas_excluded': 5},
  'issue_classifier': {'local_schemas': 2,
   'inbound_bindings': 1,
   'outbound_bindings': 1,
   'local_fixtures': 2,
   'global_schemas_excluded': 5},
  'priority_scorer': {'local_schemas': 2,
   'inbound_bindings': 1,
   'outbound_bindings': 1,
   'local_fixtures': 2,
   'global_schemas_excluded': 5},
  'customer_context_loader': {'local_schemas': 2,
   'inbound_bindings': 1,
   'outbound_bindings': 1,
   'local_fixtures': 2,
   'global_schemas_excluded': 5},
  'digest_assembler': {'local_schemas': 6,
   'inbound_bindings': 4,
   'outbound_bindings': 0,
   'local_fixtures': 3,
   'global_schemas_excluded': 1}}}

---
## Phase 5: Deterministic Component Generation

**Input:** `PacketBundle`  
**Output:** `GeneratedPackage` — one Python module per component  
**Status:** `proven`  
**Execution mode:** `live` (deterministic generator — no LLM calls)  

### Acceptance Criteria
1. A Python module is emitted for every component.
2. Every module is syntactically valid Python.
3. Every module exposes a `build_component()` callable.
4. Generation completes without exceptions.

The deterministic generator renders components from packet context and
fixture-driven templates. It is the control lane for the proof slice and
does not require a model call.

In [7]:
# INPUT
# packet_bundle  (PacketBundle from Phase 4)
generator_kind = 'deterministic'  # ← swap to 'llm' to test LLM generation

import ast
from ac14.generated_codegen import emit_generated_package

# Emit one Python module per component into a temp directory.
# 'deterministic' uses template rendering — no LLM calls, fully reproducible.
with tempfile.TemporaryDirectory() as tmpdir:
    generated_pkg = emit_generated_package(
        packet_bundle,
        output_dir=tmpdir,
        generator_kind=generator_kind,
    )

    # Syntax-check every emitted module. A SyntaxError here means the generator
    # produced broken Python — it should never happen for deterministic.
    syntax_results = {}
    build_component_present = {}
    for cid, module_path in generated_pkg.module_paths.items():
        source = open(module_path).read()
        try:
            ast.parse(source)
            syntax_results[cid] = 'ok'
        except SyntaxError as exc:
            syntax_results[cid] = f'SyntaxError: {exc}'
        # Each module must expose build_component() — that's the runtime entry point.
        build_component_present[cid] = 'def build_component' in source

assert all(v == 'ok' for v in syntax_results.values()), syntax_results
assert all(build_component_present.values()), build_component_present

phase5_result = {
    'generator_kind': generator_kind,
    'module_count': len(generated_pkg.module_paths),
    'syntax_ok': all(v == 'ok' for v in syntax_results.values()),
    'all_have_build_component': all(build_component_present.values()),
}

phase5_result  # OUTPUT: generation summary

{'generator_kind': 'deterministic',
 'module_count': 5,
 'syntax_ok': True,
 'all_have_build_component': True}

---
## Phase 6: Recomposition Proof

**Input:** `FrozenBlueprint` + reference component builders  
**Output:** `RecompositionReport` — per-scenario pass/fail with mismatch details  
**Status:** `proven`  
**Execution mode:** `live` (reference components — no LLM calls)  

### Acceptance Criteria
1. Every runnable scenario in the blueprint executes without a runtime exception.
2. All scenarios pass the categorical fixture check.
3. The report is reviewable: mismatches include component ID and field-level detail.

Reference components are the oracle/golden implementations shipped in
`ac14/reference_components.py`. They prove that the blueprint's recomposition
scenarios are correct before testing generated components.

In [8]:
# INPUT
# blueprint  (FrozenBlueprint from Phase 3)
# uses reference (oracle) component builders — swap to generated builders to test LLM output

from ac14.reference_components import build_reference_component_builders_for_blueprint
from ac14.recomposition import run_recomposition_proof

# Reference components are the hand-written oracle implementations. Running the
# proof against them confirms the blueprint's fixture scenarios are themselves
# correct before we test generated components against them.
builders = build_reference_component_builders_for_blueprint(blueprint)
recomp_report = run_recomposition_proof(blueprint, builders)

scenario_results = [
    {
        'scenario_id': r.scenario_id,
        'passed': r.passed,
        'error': r.error,
        'mismatch_component': r.mismatch_component_id,
    }
    for r in recomp_report.results
]

assert all(r.passed for r in recomp_report.results), \
    f'Recomposition failures: {[r for r in scenario_results if not r["passed"]]}'

phase6_result = {
    'total_scenarios': len(recomp_report.results),
    'passed': sum(1 for r in recomp_report.results if r.passed),
    'failed': sum(1 for r in recomp_report.results if not r.passed),
    'scenario_results': scenario_results,
}

phase6_result  # OUTPUT: per-scenario pass/fail

{'total_scenarios': 2,
 'passed': 2,
 'failed': 0,
 'scenario_results': [{'scenario_id': 'happy_path',
   'passed': True,
   'error': None,
   'mismatch_component': None},
  {'scenario_id': 'missing_customer_context',
   'passed': True,
   'error': None,
   'mismatch_component': None}]}

---
## Phase 7: Acceptance Review

**Input:** `FrozenBlueprint` + realistic input record + reference generators  
**Output:** acceptance verdict — structural checks pass/fail  
**Status:** `proven`  
**Execution mode:** `dry_run` — structural checks run live; LLM semantic review is skipped  

### Acceptance Criteria
1. The realistic input record is routed through reference components without exception.
2. The output produced by the reference system matches the expected structure.
3. Structural checks all pass.

### Promotion path
Promote to `live` once the front-half-first smoke gate produces `ready_for_full_trials`.
At that point replace reference components with generated components and enable the
LLM semantic review surface (`AC14_ENABLE_LIVE_LLM_READINESS=1`).

To run the full realistic acceptance review including LLM semantic review:
```bash
make acceptance-review \
  INPUT=examples/support_ticket_digest/blueprint \
  OUTPUT=.ac14_out/acceptance_realistic \
  GENERATOR=reference \
  REALISTIC_INPUT=examples/support_ticket_digest/input/realistic_ticket_batch.json \
  RECORD_INDEX=0
```

In [9]:
# INPUT
# blueprint  (FrozenBlueprint from Phase 3)
# uses reference components (dry_run mode — LLM semantic review skipped)

from ac14.reference_components import build_reference_component_builders_for_blueprint
from ac14.recomposition import execute_recomposition_scenarios, build_recomposition_scenario_catalog

builders = build_reference_component_builders_for_blueprint(blueprint)

# Execute every runnable scenario end-to-end through the composed system.
# In the full acceptance lane (make acceptance-review) this is followed by an
# LLM semantic review of the outputs against the stated requirements.
# Here we skip the LLM call and just prove the structural path is error-free.
catalog, executions = execute_recomposition_scenarios(blueprint, builders)

assert len(catalog.runnable_scenarios) > 0, 'No runnable scenarios — check fixtures'
assert all(e.error is None for e in executions), \
    f'Runtime errors: {[e.error for e in executions if e.error]}'

first = executions[0]
phase7_result = {
    'mode': 'dry_run',
    'runnable_scenarios': len(catalog.runnable_scenarios),
    'skipped_scenarios':  len(catalog.skipped_scenarios),
    'all_error_free':     all(e.error is None for e in executions),
    'first_scenario_id':  first.scenario_id,
    'components_with_output': list((first.outputs_by_component or {}).keys()),
    'semantic_review': 'skipped — run: make acceptance-review GENERATOR=reference',
}

phase7_result  # OUTPUT: structural acceptance verdict

{'mode': 'dry_run',
 'runnable_scenarios': 2,
 'skipped_scenarios': 1,
 'all_error_free': True,
 'first_scenario_id': 'happy_path',
 'components_with_output': ['ticket_parser',
  'issue_classifier',
  'priority_scorer',
  'customer_context_loader',
  'digest_assembler'],
 'semantic_review': 'skipped — run: make acceptance-review GENERATOR=reference'}

---
## Phase 8: Empirical Comparison Gate (Front-Half-First)

**Input:** structured-spec front-half artifact + runtime evaluation harness  
**Output:** smoke verdict — `ready_for_full_trials` / `blocked_on_*`  
**Status:** `planned`  
**Execution mode:** `planned`  

### Acceptance Criteria
1. AC14 must first produce an approved structured-spec front-half artifact.
2. Runtime hard-harness success is the end-to-end success surface.
3. Monolithic stays runtime-only — no invented monolithic front-half artifact.
4. The smoke verdict is persisted before the full five-trial budget is spent.

### Current empirical baseline

| Gate | Artifact | Verdict |
|------|----------|---------|
| Gate 1 (`order_exception_resolution_v1`) | `.ac14_out/full_trials_gate_1/experiment_decision.json` | `inconclusive` (2/5 vs 2/5) |
| Gate 2 (`resource_scaling_v1`) | `.ac14_out/full_trials_gate_2/experiment_decision.json` | `monolithic_wins` (0/5 vs 5/5) |
| Front-half-first smoke (current) | `.ac14_out/front_half_first_smoke_15/` | pending |

### Branch tree from smoke_15 verdict

```
smoke_15 verdict
  ready_for_full_trials  → Plan #88 (full trial gate) → Plan #100 (verdict interpretation)
  blocked_on_runtime_outputs → Plan #120 (boundary) → Plan #121 (repair + smoke_16)
  blocked_on_harness         → Plan #130 (boundary) → Plan #131 (repair + smoke_17)
  blocked_on_front_half      → Plan #126 (boundary) → Plan #127 (repair + smoke_15)
  blocked_on_infrastructure  → Plan #128 (boundary) → Plan #129 (fallback + smoke_16)
```

### To run the smoke gate
```bash
MODEL=gpt-5-mini make front-half-first-smoke-gate \
  BENCHMARK=benchmarks/resource_scaling_structured_spec \
  OUTPUT=.ac14_out/front_half_first_smoke_15
```

### Promotion path
This phase promotes to `live` the moment a persisted `smoke_readiness_report.json`
exists. Load it here and branch from the verdict — do not branch from expectation.

In [10]:
# INPUT
# Smoke artifact path — self-upgrades to 'live' mode once smoke_15 completes
smoke_report_path = REPO_ROOT / '.ac14_out' / 'front_half_first_smoke_15' / 'smoke_readiness_report.json'
gate1_path = REPO_ROOT / '.ac14_out' / 'full_trials_gate_1' / 'experiment_decision.json'
gate2_path = REPO_ROOT / '.ac14_out' / 'full_trials_gate_2' / 'experiment_decision.json'

# Load the two completed gate verdicts.
gate1 = json.loads(gate1_path.read_text())['verdict'] if gate1_path.exists() else 'artifact_missing'
gate2 = json.loads(gate2_path.read_text())['verdict'] if gate2_path.exists() else 'artifact_missing'

# Load smoke_15 if it exists, else emit a provisional 'pending' verdict so the
# rest of the notebook can still run.
if smoke_report_path.exists():
    smoke = json.loads(smoke_report_path.read_text())
    smoke_verdict = smoke.get('verdict', 'unknown')
    mode = 'live'
else:
    smoke_verdict = 'pending'
    mode = 'planned'

# Map each possible verdict to the next numbered plan to execute.
next_plan = {
    'ready_for_full_trials':   'Plan #88 → Plan #100',
    'blocked_on_runtime_outputs': 'Plan #120 → Plan #121',
    'blocked_on_harness':      'Plan #130 → Plan #131',
    'blocked_on_front_half':   'Plan #126 → Plan #127',
    'blocked_on_infrastructure': 'Plan #128 → Plan #129',
    'pending': 'run smoke_15 then reload this cell',
}.get(smoke_verdict, 'unknown verdict — check smoke artifact')

phase8_result = {
    'mode': mode,
    'gate1_verdict': gate1,
    'gate2_verdict': gate2,
    'smoke15_verdict': smoke_verdict,
    'next_plan': next_plan,
}

phase8_result  # OUTPUT: empirical gate state

{'mode': 'live',
 'gate1_verdict': 'inconclusive',
 'gate2_verdict': 'monolithic_wins',
 'smoke15_verdict': 'blocked_on_harness',
 'next_plan': 'Plan #130 → Plan #131'}

---
## Journey Summary

**Input:** `input_summary` → `phase2_fixture_summary` → `phase3_result` → `phase4_result` → `phase5_result` → `phase6_result` → `phase7_result` → `phase8_result`

**Output:** end-to-end verified pipeline status

In [11]:
journey_summary = {
    'phase_1_input_inspection':    {'status': 'proven',  'mode': 'live',    'records':    phase1_result['record_count']},
    'phase_2_front_half':          {'status': 'proven',  'mode': 'fixture', 'components': phase2_fixture_summary['component_count']},
    'phase_3_b1_validation':       {'status': 'proven',  'mode': 'live',    'passed':     phase3_result['b1_passed']},
    'phase_4_packet_compilation':  {'status': 'proven',  'mode': 'live',    'packets':    phase4_result['packet_count']},
    'phase_5_component_generation':{'status': 'proven',  'mode': 'live',    'syntax_ok':  phase5_result['syntax_ok']},
    'phase_6_recomposition_proof': {'status': 'proven',  'mode': 'live',    'passed':     phase6_result['passed']},
    'phase_7_acceptance_review':   {'status': 'proven',  'mode': 'dry_run', 'error_free': phase7_result['all_error_free']},
    'phase_8_empirical_gate':      {'status': 'planned', 'mode': phase8_result['mode'],   'smoke15': phase8_result['smoke15_verdict']},
}

proven  = sum(1 for p in journey_summary.values() if p['status'] == 'proven')
planned = sum(1 for p in journey_summary.values() if p['status'] == 'planned')
print(f'Proven phases:  {proven}/{len(journey_summary)}')
print(f'Planned phases: {planned}/{len(journey_summary)}')
journey_summary

Proven phases:  7/8
Planned phases: 1/8


{'phase_1_input_inspection': {'status': 'proven',
  'mode': 'live',
  'records': 3},
 'phase_2_front_half': {'status': 'proven',
  'mode': 'fixture',
  'components': 5},
 'phase_3_b1_validation': {'status': 'proven', 'mode': 'live', 'passed': True},
 'phase_4_packet_compilation': {'status': 'proven',
  'mode': 'live',
  'packets': 5},
 'phase_5_component_generation': {'status': 'proven',
  'mode': 'live',
  'syntax_ok': True},
 'phase_6_recomposition_proof': {'status': 'proven',
  'mode': 'live',
  'passed': 2},
 'phase_7_acceptance_review': {'status': 'proven',
  'mode': 'dry_run',
  'error_free': True},
 'phase_8_empirical_gate': {'status': 'planned',
  'mode': 'live',
  'smoke15': 'blocked_on_harness'}}